In [1]:
RUN_MODE = "observed-dev"  # observed-dev | production
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "observed-dev-20260806.1"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
EMPIRICAL_ANALYSIS_ALLOWED = False
PROMOTION_ALLOWED = False
PROJECT_ROOT = None
RELEASE_ROOT = None
OUTPUT_ROOT = None

# P4 Notebook-First · A2-07-LABEL

## Label observed career access

| Field | Value |
|---|---|
| Agent | `P4-A2-PIPELINE` |
| Run mode | `observed-dev` |
| Contract | `2.1.2` |
| Provenance | `OBSERVED_DEVELOPMENT_ONLY` |
| Empirical / promotion | `false / false` |

Apply development labels while preserving unresolved boundaries.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pipeline/pyproject.toml").exists() and (candidate / "pipeline/src/p4").exists():
            return candidate
    raise RuntimeError("project root not found")

PROJECT_ROOT = Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else find_project_root(Path.cwd().resolve())
PIPELINE_ROOT = PROJECT_ROOT / "pipeline"
sys.path.insert(0, str(PIPELINE_ROOT / "src"))
RELEASE_ROOT = Path(RELEASE_ROOT).resolve() if RELEASE_ROOT else PROJECT_ROOT / "crawl/observed_inputs/OBSERVED_INPUT_20260806_01"
CRAWL_ROOT = Path(os.environ.get("P4_CRAWL_ROOT", PROJECT_ROOT / "crawl")).resolve()
OUTPUT_ROOT = Path(OUTPUT_ROOT).resolve() if OUTPUT_ROOT else PIPELINE_ROOT / "data/exports/observed-dev/OBSERVED_DEV_20260806_01"
CONTROL_ROOT = Path(os.environ.get("P4_CONTROL_ROOT", PROJECT_ROOT / "crawl/control")).resolve()
NCS_PROJECT_ROOT = Path(os.environ.get("P4_NCS_PROJECT_ROOT", PROJECT_ROOT)).resolve()
NCS_HANDOFF_PATH = Path(os.environ.get("P4_NCS_HANDOFF_PATH", NCS_PROJECT_ROOT / "shared/handoffs/AGENT4_TO_AGENT2_NCS_MAPPING_OBSERVED_DEV.json")).resolve()
RUN_ROOT = Path(os.environ.get("P4_NOTEBOOK_RUN_ROOT", PIPELINE_ROOT / "runs/notebooks/observed-dev/AGENT2_20260806_01")).resolve()
BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
GIT_HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
METADATA = {
    "agentId": "P4-A2-PIPELINE", "branch": BRANCH, "gitHead": GIT_HEAD,
    "contractVersion": CONTRACT_VERSION, "crawlReleaseId": CRAWL_RELEASE_ID,
    "dataVersion": DATA_VERSION, "runMode": RUN_MODE, "dataProvenance": DATA_PROVENANCE,
    "asOfDate": AS_OF_DATE, "randomSeed": RANDOM_SEED,
    "startedAt": datetime.now(timezone.utc).isoformat(),
    "inputManifestPath": "crawl/observed_inputs/OBSERVED_INPUT_20260806_01/HANDOFF.json",
    "outputRoot": "pipeline/data/exports/observed-dev/OBSERVED_DEV_20260806_01",
    "empiricalAnalysisAllowed": EMPIRICAL_ANALYSIS_ALLOWED, "promotionAllowed": PROMOTION_ALLOWED,
    "storagePolicy": {"canonical": "DUCKDB_PARQUET", "inspectionExport": "CSV_UTF8_SIG"},
    "eligibilityColumns": ["postingEligibleFlag", "rq1EligibleFlag", "rq2EligibleFlag", "ncsEligibleFlag"],
    "highDemandScorePolicy": "ALL_NULL",
    "ksaPolicy": {"decisionId": "D-023", "status": "PROVISIONAL", "mode": "OPTIONAL_ENRICHMENT", "blocksM1": False},
    "mappingPolicy": {"mappingMode": "LEXICAL_BASELINE", "codeSetStatus": "REVIEW_REQUIRED", "goldValidatedFlag": False, "denseScore": None},
}
assert RUN_MODE == "observed-dev"
assert DATA_PROVENANCE == "OBSERVED_DEVELOPMENT_ONLY"
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
if (CONTROL_ROOT / "NOTEBOOK_EXECUTION_CONTRACT.schema.json").is_file():
    import jsonschema
    schema = json.loads((CONTROL_ROOT / "NOTEBOOK_EXECUTION_CONTRACT.schema.json").read_text(encoding="utf-8"))
    jsonschema.Draft202012Validator(schema, format_checker=jsonschema.FormatChecker()).validate(METADATA)
print(json.dumps(METADATA, ensure_ascii=False, indent=2))

{
  "agentId": "P4-A2-PIPELINE",
  "branch": "agent/p4-pipeline-v2",
  "gitHead": "90d120d2a40913b69e53a2f2db79b3abbef86ee3",
  "contractVersion": "2.1.2",
  "crawlReleaseId": "CRAWL_20260806_03",
  "dataVersion": "observed-dev-20260806.1",
  "runMode": "observed-dev",
  "dataProvenance": "OBSERVED_DEVELOPMENT_ONLY",
  "asOfDate": "2026-08-06",
  "randomSeed": 42,
  "startedAt": "2026-08-06T08:39:42.125910+00:00",
  "inputManifestPath": "crawl/observed_inputs/OBSERVED_INPUT_20260806_01/HANDOFF.json",
  "outputRoot": "pipeline/data/exports/observed-dev/OBSERVED_DEV_20260806_01",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false,
  "storagePolicy": {
    "canonical": "DUCKDB_PARQUET",
    "inspectionExport": "CSV_UTF8_SIG"
  },
  "eligibilityColumns": [
    "postingEligibleFlag",
    "rq1EligibleFlag",
    "rq2EligibleFlag",
    "ncsEligibleFlag"
  ],
  "highDemandScorePolicy": "ALL_NULL",
  "ksaPolicy": {
    "decisionId": "D-023",
    "status": "PROVISIONAL",
    "mode":

## Stage contract

**Inputs**

- `posting tracks`
- `requirement facts`
- `section boundaries`

**Outputs**

- `observed.career_access_label`
- `E and I development labels`
- `four termination artifacts`

In [3]:
from p4.notebooks.observed_stages import audit_observed_stage_inputs
STAGE = '07LabelCareerAccess'
INPUT_AUDIT = audit_observed_stage_inputs(
    STAGE,
    project_root=PROJECT_ROOT,
    release_root=RELEASE_ROOT,
    ncs_handoff_path=NCS_HANDOFF_PATH,
)
assert INPUT_AUDIT["missingInputCount"] == 0
print(json.dumps(INPUT_AUDIT, ensure_ascii=False, indent=2))

{
  "stage": "07LabelCareerAccess",
  "requiredInputCount": 2,
  "missingInputCount": 0,
  "inputNames": [
    "HANDOFF.json",
    "posting_manifest.parquet"
  ],
  "runMode": "observed-dev",
  "dataProvenance": "OBSERVED_DEVELOPMENT_ONLY",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false
}


## Execute versioned stage module

In [4]:
from p4.notebooks.observed_stages import run_label_career_access_stage
RESULT = run_label_career_access_stage(
    project_root=PROJECT_ROOT,
    release_root=RELEASE_ROOT,
    crawl_root=CRAWL_ROOT,
    output_root=OUTPUT_ROOT,
    control_root=CONTROL_ROOT,
    ncs_handoff_path=NCS_HANDOFF_PATH,
    ncs_project_root=NCS_PROJECT_ROOT,
    run_root=RUN_ROOT,
)
assert RESULT["qualityStatus"] == "PASS", RESULT
print(json.dumps(RESULT, ensure_ascii=False, indent=2))

{
  "stageManifest": "runs/notebooks/observed-dev/AGENT2_20260806_01/artifacts/07LabelCareerAccess/stage_manifest.json",
  "stageMetrics": "runs/notebooks/observed-dev/AGENT2_20260806_01/artifacts/07LabelCareerAccess/stage_metrics.json",
  "stageQuality": "runs/notebooks/observed-dev/AGENT2_20260806_01/artifacts/07LabelCareerAccess/stage_quality.csv",
  "checksums": "runs/notebooks/observed-dev/AGENT2_20260806_01/artifacts/07LabelCareerAccess/CHECKSUMS.sha256",
  "qualityStatus": "PASS"
}


## Stage summary

In [5]:
from IPython.display import display
import pandas as pd

SUMMARY = pd.DataFrame([
    {"field": "stage", "value": STAGE},
    {"field": "qualityStatus", "value": RESULT["qualityStatus"]},
    {"field": "stageManifest", "value": RESULT["stageManifest"]},
    {"field": "stageMetrics", "value": RESULT["stageMetrics"]},
    {"field": "stageQuality", "value": RESULT["stageQuality"]},
    {"field": "checksums", "value": RESULT["checksums"]},
])
display(SUMMARY)

,field,value
0,stage,07LabelCareerAccess
1,qualityStatus,PASS
2,stageManifest,runs/notebooks/observed-dev/AGENT2_20260806_01...
3,stageMetrics,runs/notebooks/observed-dev/AGENT2_20260806_01...
4,stageQuality,runs/notebooks/observed-dev/AGENT2_20260806_01...
5,checksums,runs/notebooks/observed-dev/AGENT2_20260806_01...


## Termination contract

In [6]:
REQUIRED_TERMINATION_ARTIFACTS = (
    "stage_manifest.json", "stage_metrics.json", "stage_quality.csv", "CHECKSUMS.sha256",
)
artifact_root = RUN_ROOT / "artifacts" / STAGE
missing = [name for name in REQUIRED_TERMINATION_ARTIFACTS if not (artifact_root / name).is_file()]
assert not missing, missing
assert RESULT["qualityStatus"] == "PASS"
print(json.dumps({
    "stage": STAGE,
    "terminationArtifacts": list(REQUIRED_TERMINATION_ARTIFACTS),
    "artifactRoot": str(artifact_root.relative_to(PROJECT_ROOT)),
    "empiricalAnalysisAllowed": False,
    "promotionAllowed": False,
}, ensure_ascii=False, indent=2))

{
  "stage": "07LabelCareerAccess",
  "terminationArtifacts": [
    "stage_manifest.json",
    "stage_metrics.json",
    "stage_quality.csv",
    "CHECKSUMS.sha256"
  ],
  "artifactRoot": "pipeline/runs/notebooks/observed-dev/AGENT2_20260806_01/artifacts/07LabelCareerAccess",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false
}
